# Angular distribution of the elliptic objects along the bone

This script reads the csv-files of the elliptical objects found either in the x-y or z-y plane and builds a histogram of the orientation angles of the fitted ellipsoids.

Input: csv-files of ellipsoids objects in the CD105 segmentation along the x-y and z-y plane

Output: histgrams over orientation angles

In [ ]:
# 1. Import all required libraries
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import pandas as pd

## Define standard colors and legend titles

In [ ]:
# Young mice without blood loss => shades of blue
Ycolors = ['lightblue', 'deepskyblue', 'dodgerblue', 'blue', 'navy']
Ylabels = ['VK-AA498', 'VK-AA499', 'VK-AA500', 'VK-AA501', 'VK-AA502']

# Young. blood loss mice => shades of orange
Bcolors = ['peachpuff', 'sandybrown', 'darkorange', 'chocolate', 'saddlebrown']
Blabels = ['VK-AA763', 'VK-AA764', 'VK-AA765', 'VK-AA766', 'VK-AA767']

# Old mice => shades of grey
Ocolors = ['black', 'dimgrey', 'grey', 'darkgrey', 'gainsboro']
Olabels = ['VK-AA758', 'VK-AA759', 'VK-AA760', 'VK-AA761', 'VK-AA762']

## Define datasets

In [ ]:
# Young mice 
Ymouse_1_xy = 'VK-AA498_shaft_1018-1697_x-y.csv'
Ymouse_2_xy = 'VK-AA499_shaft_885-1475_x-y.csv'
Ymouse_3_xy = 'VK-AA500_shaft_1074-1790_x-y.csv'
Ymouse_4_xy = 'VK-AA501_shaft_1018-1697_x-y.csv'
Ymouse_5_xy = 'VK-AA502_shaft_1003-1672_x-y.csv'

Ymouse_1_zy = 'VK-AA498_shaft_1018-1697_z-y.csv'
Ymouse_2_zy = 'VK-AA499_shaft_885-1475_z-y.csv'
Ymouse_3_zy = 'VK-AA500_shaft_1074-1790_z-y.csv'
Ymouse_4_zy = 'VK-AA501_shaft_1018-1697_z-y.csv'
Ymouse_5_zy = 'VK-AA502_shaft_1003-1672_z-y.csv'

In [ ]:
# Young, blood-loss mice 
Bmouse_1_xy = 'VK-AA763_shaft_918-1530_x-y.csv'
Bmouse_2_xy = 'VK-AA764_shaft_973-1622_x-y.csv'
Bmouse_3_xy = 'VK-AA765_shaft_945-1575_x-y.csv'
Bmouse_4_xy = 'VK-AA766_shaft_898-1497_x-y.csv'
Bmouse_5_xy = 'VK-AA767_shaft_928-1547_x-y.csv'

Bmouse_1_zy = 'VK-AA763_shaft_918-1530_z-y.csv'
Bmouse_2_zy = 'VK-AA764_shaft_973-1622_z-y.csv'
Bmouse_3_zy = 'VK-AA765_shaft_945-1575_z-y.csv'
Bmouse_4_zy = 'VK-AA766_shaft_898-1497_z-y.csv'
Bmouse_5_zy = 'VK-AA767_shaft_928-1547_z-y.csv'

In [ ]:
# Aged mice 
Omouse_1_xy = 'VK-AA758_shaft_1174-1957_x-y.csv'
Omouse_2_xy = 'VK-AA759_shaft_1167-1945_x-y.csv'
Omouse_3_xy = 'VK-AA760_shaft_1228-2047_x-y.csv'
Omouse_4_xy = 'VK-AA761_shaft_1150-1917_x-y.csv'
Omouse_5_xy = 'VK-AA762_shaft_1245-2075_x-y.csv'

Omouse_1_zy = 'VK-AA758_shaft_1174-1957_z-y.csv'
Omouse_2_zy = 'VK-AA759_shaft_1167-1945_z-y.csv'
Omouse_3_zy = 'VK-AA760_shaft_1228-2047_z-y.csv'
Omouse_4_zy = 'VK-AA761_shaft_1150-1917_z-y.csv'
Omouse_5_zy = 'VK-AA762_shaft_1245-2075_z-y.csv'

## Define required functions

In [ ]:
# Define functions to read all five datasets of case and to generate overlaid histograms
def read_angle_column(data: np.ndarray):
    raw_angles =  np.genfromtxt(data ,skip_header=1, delimiter=',', usecols=(9))
    #aligned_angles = np.where(raw_angles > 90, raw_angles - 180, raw_angles)
    
    return raw_angles

def plot_polar_histogram(datasets, bins=90, colors=None, labels=None, title="Polar Histogram"):
    # Create a polar plot
    fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})

    for i, data in enumerate(datasets):
        # Compute histogram
        counts, bin_edges = np.histogram(data, bins=bins)
    
        #Convert bin edges from degrees to radiants (as expected by MatplotLib!)
        bin_egdes_radiant = bin_edges * (np.pi/180)
    
        # Convert the histogram counts into a density
        density = counts / np.sum(counts)

        # Plot the histogram as line plot
        ax.plot(bin_egdes_radiant[:-1], density, color=colors[i] if colors else None, label=labels[i] if labels else None,)

    # Customize plot
    ax.set_title(title, fontsize = 12, fontweight = 'bold')
    ax.tick_params(axis='y', rotation=45)
    ax.set_thetamin(0)
    ax.set_thetamax(180)
    
    # Add legend if labels are provided
    if labels:
        plt.legend()
        
    # Show the plot
    plt.show()
    
# Function to save the density histograms
def save_density_histograms_to_csv(datasets, bins=180, csv_filename="combined_density_histograms.csv"):
    
    # Create an empty list to store histogram dataframes
    dfs = []
    
    for i, data in enumerate(datasets):
        # Compute density histogram values
        values, bin_edges = np.histogram(data, bins=bins, density=True) # Set density=False to obtain actual numbers
        
        # Create a DataFrame for the current dataset's histogram
        df = pd.DataFrame({'Bin Edges': bin_edges[:-1], f'Density Values_{i}': values})  # Add a column with density values for the current datasets
        
        # Append the DataFrame to the list
        dfs.append(df)
    
    # Merge all the individual dataframes on the 'Bin Edges' column
    merged_df = dfs[0]  # Start with the first DataFrame
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on='Bin Edges', how='outer')
    
    # Save the merged DataFrame to the specified CSV file
    merged_df.to_csv(csv_filename, index=False)

## Young mice datasets

In [ ]:
# Load young mice data sets
Ydatasets_xy = [Ymouse_1_xy, Ymouse_2_xy, Ymouse_3_xy, Ymouse_4_xy, Ymouse_5_xy]
Ydatasets_zy = [Ymouse_1_zy, Ymouse_2_zy, Ymouse_3_zy, Ymouse_4_zy, Ymouse_5_zy]
Yall_angle = list()

for dataset_xy, dataset_zy in zip(Ydatasets_xy,Ydatasets_zy):
    angles_xy = read_angle_column(dataset_xy)
    angles_zy = read_angle_column(dataset_zy)
    angles = np.concatenate([angles_xy, angles_zy])
    Yall_angle.append(angles)

#plot_polar_histogram(Yall_angle, bins=90, colors=Ycolors, labels=Ylabels, title='Young Mice (LC) - hip')

## Young, blood loss mice

In [ ]:
# Load young mice data sets
Bdatasets_xy = [Bmouse_1_xy, Bmouse_2_xy, Bmouse_3_xy, Bmouse_4_xy, Bmouse_5_xy]
Bdatasets_zy = [Bmouse_1_zy, Bmouse_2_zy, Bmouse_3_zy, Bmouse_4_zy, Bmouse_5_zy]
Ball_angle = list()

for dataset_xy, dataset_zy in zip(Bdatasets_xy,Bdatasets_zy):
    angles_xy = read_angle_column(dataset_xy)
    angles_zy = read_angle_column(dataset_zy)
    angles = np.concatenate([angles_xy, angles_zy])
    Ball_angle.append(angles)

#plot_polar_histogram(Ball_angle, bins=90, colors=Bcolors, labels=Blabels, title='Young, blood loss Mice (LC) - knee')

## Aged mice

In [ ]:
# Load aged mice data sets
Odatasets_xy = [Omouse_1_xy, Omouse_2_xy, Omouse_3_xy, Omouse_4_xy, Omouse_5_xy]
Odatasets_zy = [Omouse_1_zy, Omouse_2_zy, Omouse_3_zy, Omouse_4_zy, Omouse_5_zy]
Oall_angle = list()

for dataset_xy, dataset_zy in zip(Odatasets_xy,Odatasets_zy):
    angles_xy = read_angle_column(dataset_xy)
    angles_zy = read_angle_column(dataset_zy)
    angles = np.concatenate([angles_xy, angles_zy])
    Oall_angle.append(angles)

#plot_polar_histogram(Oall_angle, bins=90, colors=Ocolors, labels=Olabels, title='Aged Mice (LC) - knee')

## Save histograms (density)

In [ ]:
save_density_histograms_to_csv(Ball_angle, csv_filename="Blood-let(hip-mid2).csv")
save_density_histograms_to_csv(Yall_angle, csv_filename="Young(hip-mid2).csv")
save_density_histograms_to_csv(Oall_angle, csv_filename="Old(hip-mid2).csv")